In [1]:
##### SI Table 1: metrics of RF model performance

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score

In [4]:
##### SET-UP

# Get the current working directory
cd = Path.cwd().parent.parent

# import reference data 
ref_data = pd.read_csv(f"{cd}/Data/Clean/Intensities/intensities_for_eval.csv")

# import model predictions
capital_rf = pd.read_parquet(f'{cd}/Results/RF_models_final/capital_rf_spatial_CV/predictions.parquet')
labor_rf = pd.read_parquet(f'{cd}/Results/RF_models_final/labor_rf_spatial_CV/predictions.parquet')

capital_qrf = pd.read_parquet(f'{cd}/Results/RF_models_final/capital_qrf_spatial_CV/predictions.parquet')
labor_qrf = pd.read_parquet(f'{cd}/Results/RF_models_final/labor_qrf_spatial_CV/predictions.parquet')

In [5]:
##### PREP DATA

# Add original intensity data (sub-national and country) for evaluation
capital_rf = capital_rf.merge(ref_data, on=['PROJ_ID', 'country_ID'], how='left')
labor_rf = labor_rf.merge(ref_data, on=['PROJ_ID', 'country_ID'], how='left')
capital_qrf = capital_qrf.merge(ref_data, on=['PROJ_ID', 'country_ID'], how='left')
labor_qrf = labor_qrf.merge(ref_data, on=['PROJ_ID', 'country_ID'], how='left')

# Estimate actual predicted intensity (not relative)
capital_rf['predicted_log_capital_intensity_USD_per_million_tonne'] = capital_rf['prediction'] + capital_rf['log_country_capital_intensity_USD_per_million_tonne']
capital_rf['predicted_capital_intensity_USD_per_million_tonne'] = np.expm1(capital_rf['predicted_log_capital_intensity_USD_per_million_tonne'])
capital_qrf['q10_predicted_log_capital_intensity_USD_per_million_tonne'] = capital_qrf['q10'] + capital_qrf['log_country_capital_intensity_USD_per_million_tonne']
capital_qrf['q90_predicted_log_capital_intensity_USD_per_million_tonne'] = capital_qrf['q90'] + capital_qrf['log_country_capital_intensity_USD_per_million_tonne']

labor_rf['predicted_log_labor_intensity_jobs_per_million_tonne'] = labor_rf['prediction'] + labor_rf['log_country_labor_intensity_jobs_per_million_tonne']
labor_rf['predicted_labor_intensity_jobs_per_million_tonne'] = np.expm1(labor_rf['predicted_log_labor_intensity_jobs_per_million_tonne'])
labor_qrf['q10_predicted_log_labor_intensity_jobs_per_million_tonne'] = labor_qrf['q10'] + labor_qrf['log_country_labor_intensity_jobs_per_million_tonne']
labor_qrf['q90_predicted_log_labor_intensity_jobs_per_million_tonne'] = labor_qrf['q90'] + labor_qrf['log_country_labor_intensity_jobs_per_million_tonne']

In [8]:
##### DEFINE FUNCTIONS

def compute_fold_metrics(df, target_col, pred_col):
    """Per-fold R2 and PBIAS (% bias) for a given target/prediction column pair."""
    rows = []
    for fold, g in df.groupby("fold"):
        y_true = g[target_col]
        y_pred = g[pred_col]
        resid = y_true - y_pred
        rows.append({
            "fold": fold,
            "R2": r2_score(y_true, y_pred),
            "PBIAS": resid.mean() / y_true.mean() * 100 if y_true.mean() != 0 else np.nan,
        })
    return pd.DataFrame(rows)

def compute_fold_coverage(df, target_col, q10_col, q90_col):
    """Per-fold QRF coverage (fraction of true values falling within [q10, q90])."""
    rows = []
    for fold, g in df.groupby("fold"):
        y_true = g[target_col]
        low = g[q10_col]
        high = g[q90_col]
        covered = (y_true >= low) & (y_true <= high)
        rows.append({"fold": fold, "coverage": covered.mean()})
    return pd.DataFrame(rows)

def summarize_spec(df, target_col, pred_col, qrf_df=None, q10_col=None, q90_col=None, report_pbias=True):
    """Mean/variance of R2 across folds, plus PBIAS mean/variance when report_pbias=True
    (left as NaN otherwise, e.g. for relative-target specifications where percent bias
    isn't a meaningful quantity), plus mean QRF coverage if provided."""
    fold_metrics = compute_fold_metrics(df, target_col, pred_col)
    result = {
        "R2": fold_metrics["R2"].mean(),
        "VAR_R2": fold_metrics["R2"].var(),
        "PBIAS": fold_metrics["PBIAS"].mean() if report_pbias else np.nan,
        "VAR_PBIAS": fold_metrics["PBIAS"].var() if report_pbias else np.nan,
    }
    if qrf_df is not None:
        fold_cov = compute_fold_coverage(qrf_df, target_col, q10_col, q90_col)
        result["QRF_coverage"] = fold_cov["coverage"].mean()
    else:
        result["QRF_coverage"] = np.nan
    return result

In [9]:
##### CALCULATE METRICS

# --- test splits only (this is model evaluation) ---
capital_test_rf  = capital_rf[capital_rf["split"] == "test"]
labor_test_rf    = labor_rf[labor_rf["split"] == "test"]
capital_test_qrf = capital_qrf[capital_qrf["split"] == "test"]
labor_test_qrf   = labor_qrf[labor_qrf["split"] == "test"]

rows = []

# --- CAPITAL ---
rows.append({
    "Model": "Capital intensity", "Specification": "Relative",
    **summarize_spec(
        capital_test_rf, "rtv_log_capital_intensity_USD_per_million_tonne", "prediction",
        qrf_df=capital_test_qrf, q10_col="q10", q90_col="q90",
        report_pbias=False
    )
})
rows.append({
    "Model": "Capital intensity", "Specification": "Absolute",
    **summarize_spec(
        capital_test_rf,
        "log_region_capital_intensity_USD_per_million_tonne",
        "predicted_log_capital_intensity_USD_per_million_tonne",
        qrf_df=capital_test_qrf,
        q10_col="q10_predicted_log_capital_intensity_USD_per_million_tonne",
        q90_col="q90_predicted_log_capital_intensity_USD_per_million_tonne"
    )
})
rows.append({
    "Model": "Capital intensity", "Specification": "Null model",
    **summarize_spec(
        capital_test_rf,
        "log_region_capital_intensity_USD_per_million_tonne",
        "log_country_capital_intensity_USD_per_million_tonne"
    )
})

# --- LABOR ---
rows.append({
    "Model": "Labor intensity", "Specification": "Relative",
    **summarize_spec(
        labor_test_rf, "rtv_log_labor_intensity_jobs_per_million_tonne", "prediction",
        qrf_df=labor_test_qrf, q10_col="q10", q90_col="q90",
        report_pbias=False
    )
})
rows.append({
    "Model": "Labor intensity", "Specification": "Absolute",
    **summarize_spec(
        labor_test_rf,
        "log_region_labor_intensity_jobs_per_million_tonne",
        "predicted_log_labor_intensity_jobs_per_million_tonne",
        qrf_df=labor_test_qrf,
        q10_col="q10_predicted_log_labor_intensity_jobs_per_million_tonne",
        q90_col="q90_predicted_log_labor_intensity_jobs_per_million_tonne"
    )
})
rows.append({
    "Model": "Labor intensity", "Specification": "Null model",
    **summarize_spec(
        labor_test_rf,
        "log_region_labor_intensity_jobs_per_million_tonne",
        "log_country_labor_intensity_jobs_per_million_tonne"
    )
})

summary_full = pd.DataFrame(rows)[
    ["Model", "Specification", "R2", "VAR_R2", "PBIAS", "VAR_PBIAS", "QRF_coverage"]
]

##### PRINT TABLE

print("\n── FULL MODEL EVALUATION SUMMARY ─────────────────────────────────────────")
print(summary_full.round(3).to_string(index=False))


── FULL MODEL EVALUATION SUMMARY ─────────────────────────────────────────
            Model Specification     R2  VAR_R2  PBIAS  VAR_PBIAS  QRF_coverage
Capital intensity      Relative  0.246   0.072    NaN        NaN         0.753
Capital intensity      Absolute  0.376   0.003 -0.077      1.713         0.753
Capital intensity    Null model -0.004   0.088  1.529      2.111           NaN
  Labor intensity      Relative  0.520   0.049    NaN        NaN         0.823
  Labor intensity      Absolute  0.744   0.004 -0.494      3.869         0.822
  Labor intensity    Null model  0.287   0.170  3.188      7.672           NaN
